# 10. 오케스트레이터와 워커

오케스트레이션은 분해, 병렬 실행, 통합으로 구성된다. 분해가 틀리면 올바른 결과를 만들 수 없고, 통합이 없으면 결과가 조각으로 남는다.

### 공통 설정

`1`과 같은 블록이다. 워커를 스레드로 병렬 실행하므로 `keep`이 여러 스레드에서 불린다.

In [1]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
LOWER = os.environ["OPENAI_LOWER_MODEL"]
client = OpenAI()

PRICES = {MODEL: (2.50, 15.00), LOWER: (0.75, 4.50)}
USAGE = []

def keep(resp):
    USAGE.append((resp.model, resp.usage))  # CPython의 list.append는 스레드 간 호출에서 원자적이다
    return resp

### 코어 블록

`9`의 코어 블록이다. 워커의 도구는 문서 읽기 하나이고, `CONFIG`가 곧 워커의 설정이다.

In [3]:
CORPUS = Path("../../../../../data/corpus/graph")


def read_doc(title):
    f = CORPUS / f"{title}.md"
    if not f.exists():
        raise FileNotFoundError(f"'{title}' 문서가 없다")
    return f.read_text(encoding="utf-8")


IMPL = {"read_doc": read_doc}
TOOLS = [{"type": "function", "name": "read_doc", "strict": True,
          "description": "문서 한 편의 본문 전체를 돌려준다.",
          "parameters": {"type": "object",
                         "properties": {"title": {"type": "string", "description": "문서 제목"}},
                         "required": ["title"], "additionalProperties": False}}]

CONFIG = {"model": MODEL, "instructions": "도구로 읽은 것만 근거로 삼는다. 세 문장 안에서 답한다.",
          "tools": TOOLS, "impl": IMPL, "max_turns": 5, "max_tokens": 80000, "max_seconds": 180,
          "gated": [], "gate_error": "error: 승인이 필요하다.", "tool_error": "error: {e}"}

`run`은 여러 스레드에서 호출되므로 실행별 상태를 함수 안에 두고, 설정은 인자로만 읽는다.
`9`의 코어에서 바뀐 것은 트레이스의 `오류` 하나다. 도구 결과가 `error:`로 시작하면 오류로 센다. 오류 문구 둘이 그렇게 시작하도록 `CONFIG`에 정해 두었다.
워커 하나가 멈추면 `pool.map` 전체가 그 워커를 기다리므로 시간 예산도 남겨 둔다.

In [4]:
def call_tool(name, args, cfg, approve):
    if name in cfg["gated"] and not approve:
        return cfg["gate_error"]
    try:
        return json.dumps(cfg["impl"][name](**args), ensure_ascii=False)
    except Exception as e:
        return cfg["tool_error"].format(e=e)


def run(task, cfg=CONFIG, approve=False):
    messages = [{"role": "user", "content": task}]
    trace, t0, used = [], time.perf_counter(), 0
    for turn in range(1, cfg["max_turns"] + 1):
        r = keep(client.responses.create(model=cfg["model"], instructions=cfg["instructions"],
                                         input=messages, tools=cfg["tools"], store=False))
        used += r.usage.total_tokens
        calls = [i for i in r.output if i.type == "function_call"]
        trace.append({"턴": turn, "도구": [c.name for c in calls], "오류": 0, "토큰": used,
                      "초": round(time.perf_counter() - t0, 1)})
        if not calls:
            return r.output_text, "완료", trace
        if used > cfg["max_tokens"]:
            return None, "토큰 예산", trace
        if time.perf_counter() - t0 > cfg["max_seconds"]:
            return None, "시간 예산", trace
        outs = [call_tool(c.name, json.loads(c.arguments), cfg, approve) for c in calls]
        trace[-1]["오류"] = sum(o.startswith("error:") for o in outs)
        messages += [i.model_dump() for i in r.output]
        messages += [{"type": "function_call_output", "call_id": c.call_id, "output": o}
                     for c, o in zip(calls, outs)]
    return None, "반복 예산", trace

워커 결과는 `(답, 정지 사유, 트레이스)` 그대로 둔다. 성공 여부, 토큰, 오류 횟수는 트레이스에서 읽는다. 답이 있어도 도구 오류가 있으면 성공으로 치지 않는다.

In [5]:
def tokens(result):
    return result[2][-1]["토큰"]


def errors(result):
    return sum(t["오류"] for t in result[2])


def ok(result):
    return result[1] == "완료" and errors(result) == 0

### 구조화된 과제 분해

하위 과제를 워커에 직접 전달할 수 있도록 목록 형태의 구조화 출력으로 받는다.

In [6]:
class Subtask(BaseModel):
    n: int
    doc: str
    ask: str


class Split(BaseModel):
    subtasks: list[Subtask]


GOAL = ("한인애국단, 윤봉길, 이봉창_의거, 김도연_(정치인), 조선민족혁명당 다섯 문서를 각각 살펴"
        " 1930년대 항일 의거에서 각 문서가 무엇을 말하는지 정리한다")

분해는 하위 모델로 한다. 항목 수가 문서 수와 같은지 여기서 확인한다.

In [7]:
split = keep(client.responses.parse(
    model=LOWER, store=False, text_format=Split,
    input=f"다음 과제를 문서 한 편씩 맡는 하위 과제로 나눈다.\n\n{GOAL}")).output_parsed

for s in split.subtasks:
    print(f"{s.n}. {s.doc:16} {s.ask[:50]}")

1. 한인애국단            1930년대 항일 의거와 관련해 이 문서가 설명하는 조직의 성격, 활동 목적, 주요 의거 
2. 윤봉길              1930년대 항일 의거와 관련해 이 문서가 설명하는 윤봉길의 의거 내용, 배경, 결과, 역
3. 이봉창_의거           1930년대 항일 의거와 관련해 이 문서가 설명하는 이봉창의 의거 내용, 배경, 결과, 역
4. 김도연_(정치인)        1930년대 항일 의거와 관련해 이 문서가 설명하는 김도연의 역할이나 연관성, 관련 사건·
5. 조선민족혁명당          1930년대 항일 의거와 관련해 이 문서가 설명하는 당의 성격, 활동, 의거와의 연계, 관


### 순차와 병렬

워커 다섯을 차례로 돌린다.

In [8]:
def worker_task(s):
    return f"{s.doc} 문서를 읽고 다음에 답한다. {s.ask}"


t0 = time.perf_counter()
serial = [run(worker_task(s)) for s in split.subtasks]
serial_sec = time.perf_counter() - t0
print(f"순차 {serial_sec:5.1f}초")

순차  93.4초


같은 워커를 스레드 풀에서 동시에 돌린다.

In [9]:
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    parallel = list(pool.map(lambda s: run(worker_task(s)), split.subtasks))
parallel_sec = time.perf_counter() - t0
print(f"병렬 {parallel_sec:5.1f}초")

병렬  45.3초


독립적인 워커를 병렬로 실행하면 토큰 사용량은 같고 시간만 줄어든다.

In [10]:
print(f"순차 {serial_sec:5.1f}초   토큰 {sum(tokens(r) for r in serial):,}")
print(f"병렬 {parallel_sec:5.1f}초   토큰 {sum(tokens(r) for r in parallel):,}")
print(f"시간 {parallel_sec / serial_sec:.0%}")

순차  93.4초   토큰 27,275
병렬  45.3초   토큰 27,251
시간 49%


### 워커에 전달할 컨텍스트

전체 과제를 전달하면 워커가 맥락을 얻는 대신 실행별 컨텍스트가 커진다. 같은 하위 과제를 두 방식으로 돌린다.

In [11]:
one = split.subtasks[0]
narrow = run(worker_task(one))
wide = run(f"전체 과제는 다음과 같다.\n{GOAL}\n\n"
           f"다른 워커들이 나머지를 맡는다. 너는 이것만 한다. {worker_task(one)}")

In [12]:
print(f"자기 몫만  {tokens(narrow):6,d} 토큰   {str(narrow[0])[:60]}")
print(f"전체 맥락  {tokens(wide):6,d} 토큰   {str(wide[0])[:60]}")
print(f"\n맥락을 준 값 {tokens(wide) - tokens(narrow):+,d} 토큰")

자기 몫만   4,727 토큰   한인애국단은 1931년 상하이에서 김구가 대한민국 임시정부 내에 결성한 항일무장투쟁·암살 비밀 성격의 독립운
전체 맥락   4,917 토큰   한인애국단은 1931년 상하이에서 김구가 대한민국 임시정부 내에 조직한 항일무장투쟁 단체로, 일본 제국의 주

맥락을 준 값 +190 토큰


워커 수가 늘면 같은 맥락을 반복해서 전송하는 비용도 커진다.

워커가 자기 과제만으로 판단할 수 있으면 추가 맥락을 주지 않는다. 범위나 표기처럼 경계에 걸친 판단은 오케스트레이터가 결정해 지시한다.

### 부분 실패

하위 과제 하나를 없는 문서로 바꿔 돌린다.

In [13]:
broken = list(split.subtasks[:4]) + [Subtask(n=5, doc="없는문서", ask="무엇을 말하는가")]

with ThreadPoolExecutor(max_workers=5) as pool:
    got = list(pool.map(lambda s: run(worker_task(s)), broken))

워커별 트레이스에서 성공 여부와 도구 오류 횟수를 읽는다. 실패한 워커는 턴마다 무엇을 불렀는지도 본다.

In [14]:
for s, r in zip(broken, got):
    print(f"{'ok ' if ok(r) else '실패'} 도구오류 {errors(r)}  {s.doc:16} {str(r[0])[:40]}")

for s, r in zip(broken, got):
    if not ok(r):
        print(f"\n{s.doc} 트레이스")
        for t in r[2]:
            print(f"  턴 {t['턴']}  도구 {t['도구']}  오류 {t['오류']}")

ok  도구오류 0  한인애국단            한인애국단은 1931년 상하이에서 김구가 대한민국 임시정부 내에 결성한 
ok  도구오류 0  윤봉길              문서에 따르면 윤봉길의 의거는 1932년 4월 29일 상하이 홍커우 공원
ok  도구오류 0  이봉창_의거           이 문서는 이봉창 의거를 1932년 1월 8일 도쿄 사쿠라다문에서 한인애
ok  도구오류 0  김도연_(정치인)        이 문서에서 1930년대 항일 의거와 직접 연결해 서술되는 김도연의 역할
실패 도구오류 1  없는문서             `없는문서`라는 제목의 문서는 찾을 수 없었다.  
따라서 그 문서가 무

없는문서 트레이스
  턴 1  도구 ['read_doc']  오류 1
  턴 2  도구 []  오류 0


### 정상 응답으로 보이는 실패

도구 오류 뒤에도 모델은 정상적인 문장을 반환할 수 있다. 정지 사유만 보면 이 실패를 구분할 수 없으므로 `run`이 턴마다 도구 오류 횟수를 트레이스에 남긴다.

결과 평가만으로는 답변 생성 여부만 보이지만, 궤적 평가에서는 근거 문서를 읽지 못한 실행을 확인할 수 있다.

In [16]:
def retry(s, times=2):
    for _ in range(times):
        r = run(worker_task(s))
        if ok(r):
            return r
    return r

실패한 항목을 고친 제목으로 다시 돌린다.

In [17]:
fixed = retry(Subtask(n=5, doc="조선민족혁명당", ask="무엇을 말하는가"))
print("재시도 |", ok(fixed), str(fixed[0])[:64])

재시도 | True 조선민족혁명당 문서는 이 단체가 상해 임시정부 주변의 여러 독립운동 단체들이 일제에 맞서기 위해 통일전선을 이루며 


### 통합

통합 단계에는 워커가 줄인 결과만 전달해 컨텍스트가 다시 커지지 않도록 한다. 성공한 조각과 빠진 문서를 가른다.

In [18]:
pieces = [f"[{s.doc}] {r[0]}" for s, r in zip(broken, got) if ok(r)]
missing = [s.doc for s, r in zip(broken, got) if not ok(r)]
print(len(pieces), "조각   빠진 것", missing)

4 조각   빠진 것 ['없는문서']


In [23]:
pieces

['[한인애국단] 한인애국단은 1931년 상하이에서 김구가 대한민국 임시정부 내에 결성한 항일무장투쟁·암살 비밀 성격의 독립운동단체로, 일본 제국의 주요 요인을 처단하는 조직이었다. 활동 목적은 조국의 독립과 자유 회복을 위해 일본 제국 핵심 인물과 침략 책임자들을 암살·공격하는 데 있었고, 이를 위해 미주 동포 자금 모집과 단원 선발, 폭탄 입수·파견 등을 추진했다. 주요 의거 지원·주도 내용으로는 이봉창의 1932년 도쿄 일왕 폭탄 투척 의거와 윤봉길의 1932년 상하이 훙커우 공원 의거를 직접 조직·실행했으며, 그 밖에 이즈모호 폭파 계획, 조선총독 암살 시도, 다롄 의거 추진 등도 전개했다.',
 '[윤봉길] 문서에 따르면 윤봉길의 의거는 1932년 4월 29일 상하이 홍커우 공원에서 일본 천황 생일연과 상하이 점령 전승 기념식 중 일본인으로 위장해 수통 폭탄을 단상에 투척한 것으로, 시라카와 요시노리 등 일본 군정 핵심 인물들을 처단하거나 중상 입히고 현장에서 체포된 사건이다. 배경으로는 그가 1930년 중국으로 망명한 뒤 상하이에서 김구를 찾아 한인애국단에 가입했고, 김구가 일본의 기념행사를 폭탄으로 공격할 계획을 세워 윤봉길이 실행을 맡았다는 점이 제시된다. 결과와 역사적 의미로는 윤봉길이 사형당했지만 그의 의거가 장제스의 임시정부 지원을 이끄는 도화선이 되었고, 임시정부의 위상을 높였으며, 이승만의 평가 인용에 따르면 훗날 카이로 회담에서 한국 독립 문제가 제기된 원인의 하나로까지 연결된다고 설명한다.',
 '[이봉창_의거] 이 문서는 이봉창 의거를 1932년 1월 8일 도쿄 사쿠라다문에서 한인애국단 소속 이봉창이 일본 천황을 향해 수류탄을 던진 사건으로 설명하며, 폭탄은 천황을 죽이지 못해 의거는 실패했고 이봉창은 체포·재판 뒤 1932년 10월 10일 처형되었다고 서술한다. 배경으로는 일제 식민지 지배와 3·1 운동 이후의 독립운동, 만주사변 전후의 동아시아 정세, 그리고 일본에서 차별을 겪은 이봉창이 상하이로 가 김구와 대한민국 임시정부, 

빠진 것은 빠졌다고 적게 한다. 통합 입력이 워커 입력의 합보다 작은지 본다.

In [19]:
merged = keep(client.responses.create(
    model=MODEL, store=False,
    instructions="주어진 조각만 근거로 삼는다. 빠진 것은 빠졌다고 적는다.",
    input="조각\n" + "\n\n".join(pieces)
          + (f"\n\n얻지 못한 문서: {missing}" if missing else "")
          + f"\n\n과제\n{GOAL}"))

print(f"통합 입력 {merged.usage.input_tokens:,} 토큰")
print(merged.output_text[:400])

통합 입력 1,116 토큰
주어진 조각에 근거해만 정리하면 다음과 같습니다.  
※ **조선민족혁명당** 문서는 제공되지 않았으므로 **빠졌다고 적습니다**.

---

## 1. 한인애국단
- 1931년 상하이에서 **김구가 대한민국 임시정부 내에 결성한 항일 비밀 독립운동단체**로 설명된다.
- 성격은 **항일무장투쟁·암살 중심 조직**이며, 일본 제국의 주요 인물을 처단하는 것을 목표로 했다.
- 목적은 **조국의 독립과 자유 회복**을 위해 일본 제국 핵심 인물과 침략 책임자를 암살·공격하는 데 있었다.
- 이를 위해 **미주 동포 자금 모집, 단원 선발, 폭탄 입수와 파견** 등을 추진했다고 한다.
- 1930년대 항일 의거와 관련해서는
  - **이봉창의 1932년 도쿄 일왕 폭탄 투척 의거**
  - **윤봉길의 1932


### 단계별 실패 조건

| 단계 | 실패 | 처리 |
|---|---|---|
| 분해 | 하위 과제가 겹치거나 빠진다 | 구조화 출력으로 받아 항목 수를 확인한다 |
| 실행 | 하나의 실패가 전체 실패로 이어진다 | 워커별 트레이스에 실패를 기록한다 |
| 통합 | 결과를 이어 붙여 컨텍스트가 다시 커진다 | 축약된 결과만 받고 누락 항목을 명시한다 |

분해 오류는 이후 단계에서 복구하기 어려우므로 실행 전에 확인해야 한다.

### 하나로 잇기

앞에서 단계별로 떼어 본 분해, 병렬 실행, 통합을 함수 하나로 잇는다. 위 표의 처리가 단계마다 한 줄씩 들어 있다.
분해 항목 수가 맞지 않으면 워커를 돌리기 전에 멈춘다. 실패한 워커의 결과는 조각에서 빼고 빠진 문서로 통합에 넘긴다.

In [20]:
def orchestrate(goal, expect):
    # 분해
    subs = keep(client.responses.parse(
        model=LOWER, store=False, text_format=Split,
        input=f"다음 과제를 문서 한 편씩 맡는 하위 과제로 나눈다.\n\n{goal}")).output_parsed.subtasks
    if len(subs) != expect:
        return None, {"분해": [s.doc for s in subs]}
    # 실행
    with ThreadPoolExecutor(max_workers=len(subs)) as pool:
        results = list(pool.map(lambda s: run(worker_task(s)), subs))
    pieces = [f"[{s.doc}] {r[0]}" for s, r in zip(subs, results) if ok(r)]
    missing = [s.doc for s, r in zip(subs, results) if not ok(r)]
    # 통합
    merged = keep(client.responses.create(
        model=MODEL, store=False,
        instructions="주어진 조각만 근거로 삼는다. 빠진 것은 빠졌다고 적는다.",
        input="조각\n" + "\n\n".join(pieces)
              + (f"\n\n얻지 못한 문서: {missing}" if missing else "")
              + f"\n\n과제\n{goal}"))
    return merged.output_text, {"분해": [s.doc for s in subs], "빠진 것": missing,
                                "워커 토큰": sum(tokens(r) for r in results),
                                "통합 입력": merged.usage.input_tokens}

In [22]:
answer, info = orchestrate(GOAL, expect=5)
print(info)
print(answer if answer else "분해에서 멈췄다")

{'분해': ['한인애국단', '윤봉길', '이봉창_의거', '김도연_(정치인)', '조선민족혁명당'], '빠진 것': [], '워커 토큰': 27249, '통합 입력': 1363}
다섯 문서가 **1930년대 항일 의거**에 대해 말하는 내용은 다음과 같이 정리할 수 있다.

1. **한인애국단**
   - 1931년 상하이에서 김구가 대한민국 임시정부 내에 조직한 **항일무장투쟁 단체**로 설명된다.
   - 목적은 일본 제국의 주요 인물을 암살해 **조국의 독립과 자유를 회복**하는 데 있었다.
   - 1930년대 항일 의거와 관련해서는 **이봉창의 도쿄 의거**, **윤봉길의 상하이 의거**를 직접 추진한 핵심 조직으로 제시된다.
   - 그 밖에도 일본군 함선 폭파 계획, 조선총독 암살 시도, 다롄 의거 미수 등 여러 거사를 추진했다고 한다.
   - 특히 1932년의 두 의거를 통해 국제적 반향을 일으키고, 중국 측의 인식 변화와 김구·임시정부의 위상 강화에 기여한 단체로 서술된다.

2. **윤봉길**
   - 윤봉길은 중국으로 건너가 항일투쟁에 참여한 독립운동가로 소개된다.
   - 1932년 김구가 이끄는 **한인애국단에 가입**한 뒤, **4월 29일 홍커우 공원 의거**를 일으켰다고 설명한다.
   - 그는 일본의 천장절 및 상하이 점령 전승 기념식장에서 폭탄을 던져 일본 군정 고위 인사들에게 큰 타격을 주었다.
   - 현장에서 체포되어 사형선고를 받고 1932년 12월 일본 가나자와에서 순국했다고 한다.
   - 이 의거는 중국 측에 큰 충격과 인상을 주어 **대한민국 임시정부 지원의 계기**가 되었고, 1930년대 항일 의거의 상징적 사건으로 평가된다고 정리된다.

3. **이봉창_의거**
   - 이 문서는 이봉창이 김구의 **한인애국단**에 참여해 일본 천황을 직접 공격하려 한 배경을 설명한다.
   - 1932년 1월 8일 도쿄 **사쿠라다문**에서 천황 행렬을 향해 수류탄을 던졌으나, 폭발력이 약하고 목표도 빗나가 **암살은 

### 비용 확인

In [ ]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += u.input_tokens / 1e6s * p[0] + u.output_tokens / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")

호출 50회
입력 115,243 토큰   출력 8,804 토큰
누적 $0.4136
